[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quadrics/gaussian/gaussian.ipynb)

# A Gaussian from Point Moments

This example fits a Gaussian to a cloud of points and draws its one-sigma ellipse. Usually that takes two steps: find the mean, then the spread of the points about it, measured along the cloud's own axes. Here neither the centre nor the axes are found first. One average over the points holds both the cloud's location and its spread, and its inverse measures, at any point of the plane, how many standard deviations away from the cloud it lies.

Points are antivectors of projective geometric algebra and lines vectors, and each point's dyad, `point * (Plane & point)`, is a map from lines to points. Averaged over the cloud it is the second moment, homogeneous, so it holds the mean as well; its inverse, the precision, is a map from points to lines, and at a unit-weight point it gives one plus the squared Mahalanobis distance, `precision(point) & point`. The Gaussian is its exponential, and the one-sigma ellipse is a conic, the precision less twice the dyad of the plane of unit weight, which the precision itself recovers from the data.

In the notation of statistics the second moment reads as the augmented matrix $\begin{pmatrix}\Sigma + \boldsymbol\mu\boldsymbol\mu^{\mathsf T} & \boldsymbol\mu \\ \boldsymbol\mu^{\mathsf T} & 1\end{pmatrix}$, whose inverse evaluates at $(\mathbf x, 1)$ to $1 + (\mathbf x - \boldsymbol\mu)^{\mathsf T}\Sigma^{-1}(\mathbf x - \boldsymbol\mu)$.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
import numpy as np

from examples.animation import save_figure
from examples.quadrics.gaussian import render
from examples.quadrics.gaussian.core import Plane, Point, Scalar, mv

np.set_printoptions(precision=4, suppress=True)

## 1. A stretched, moved cloud

The cloud is drawn stretched along x, then turned and moved as a batch, so that neither its centre nor its axes are where the coordinates are.

In [ ]:
# The cloud's size, its spread along x and y before it is moved, and the seed of its samples; how it is
# moved, a translation and a turn as generator coefficients; the pixels the fit is drawn on.
n_points, spread, seed = 400, np.array([1.5, 0.5]), 4
shift, turn = (0.6, 0.3), 0.3
n_across, box = 320, np.array([[-5.0, -5.0], [6.0, 4.0]])

In [ ]:
placement = (mv.xw * shift[0] - mv.yw * shift[1]).exp() * (mv.xy * turn).exp()   # [] Motor
points = placement >> mv.point(np.random.default_rng(seed).normal(size=(n_points, 2)) * spread)   # [points] Point
pixels = mv.point(np.stack(np.meshgrid(np.linspace(box[0, 0], box[1, 0], n_across),
                                       np.linspace(box[0, 1], box[1, 1], n_across)), axis=-1))   # [rows, columns] Point

## 2. The fit

The mean of the points' dyads is the second moment, and its inverse the precision. At a unit-weight point the precision gives one plus the squared Mahalanobis distance, so the Gaussian, with peak one, is the exponential of minus half of what is left. The plane of unit weight is the precision's image of the cloud's mean, the one plane that pairs to one with every unit-weight point; subtracting its dyad twice from the precision moves the zero of the form to Mahalanobis distance one: the one-sigma ellipse as a conic, a map from points to lines.

In [ ]:
def fit_gaussian(points: Point, samples: Point) -> tuple[Scalar, Scalar]:
    """Fit unit-weight points and evaluate the Gaussian and its 1σ level set.

    Samples are unit-weight query points in the same affine chart. The returned
    density has peak one; the level vanishes at Mahalanobis distance one.
    """
    # Each point contributes its dyad, a rank-one map from planes to points; their mean holds both the
    # cloud's location and its spread, without centering.
    moment = (points * (Plane & points)).mean(axis=0)
    precision = moment.inverse()

    # The inverse moment evaluates to one plus the squared Mahalanobis distance.
    # Removing the constant gives a Gaussian with peak density one.
    squared_distance = (precision(samples) & samples) - 1
    density = (-0.5 * squared_distance).exp()

    # Recover the weight plane from the data: weight & samples == 1 for every sample.
    # Its dyad evaluates to that constant squared; subtract it twice to make Mahalanobis
    # distance one the zero locus of a Plane <- Point polarity.
    weight = precision(points.mean(axis=0))
    quadric = precision - 2 * weight * (weight & Point)
    level = quadric(samples) & samples
    return density, level


density, level = fit_gaussian(points, pixels)                                  # [rows, columns] Scalar each
save_figure(render.draw_gaussian(points, pixels, density, level), "gaussian");

In [ ]:
# The same density at the cloud's mean, and the level there: peak one, and minus one, inside the ellipse.
at_mean, level_at_mean = fit_gaussian(points, points.mean(axis=0))
print("density at the mean:", at_mean.to_array(), "  level at the mean:", level_at_mean.to_array())

In [ ]:
# checks
# The density peaks at one at the cloud's mean, and the level is the squared Mahalanobis distance less
# one, so the ellipse is the contour where the density falls to exp(-1/2).
np.testing.assert_allclose(at_mean.to_array(), 1.0, atol=1e-12)
np.testing.assert_allclose(level.to_array(), -2.0 * np.log(density.to_array()) - 1.0, atol=1e-8)